# How many questions does a knowledge-graph diagnostic need, and what does one slip cost?

A small simulation of adaptive placement on a prerequisite graph, in the spirit of knowledge-space diagnostics (e.g. the Math Academy approach of using the knowledge graph to estimate a student's knowledge profile quickly).

**Setup (synthetic, not any real curriculum):** 150 topics in 10 layers. Each topic has 1-3 prerequisites. A student's true knowledge is *downward closed*: if they know a topic, they know its prerequisites. Each question asks about one topic.

**Inference rule:** a correct answer marks the topic and all its ancestors as known; a wrong answer marks the topic and all its descendants as unknown.

**Question:** how many questions does this save versus testing every topic, and how fragile is it when students sometimes slip (miss a topic they know) or guess (get one right they don't)?

In [1]:
import numpy as np
rng=np.random.default_rng(0)

def make_graph(layers=10,width=15,rng=rng):
    N=layers*width; prereq=[[] for _ in range(N)]
    for L in range(1,layers):
        for j in range(width):
            t=L*width+j
            k=rng.integers(1,4)
            prereq[t]=list(rng.choice(np.arange((L-1)*width,L*width),size=k,replace=False))
            if L>=2 and rng.random()<0.3:
                prereq[t].append(int(rng.integers((L-2)*width,(L-1)*width)))
    N=len(prereq); anc=[set() for _ in range(N)]
    for t in range(N):
        for p in prereq[t]: anc[t]|={p}|anc[p]
    desc=[set() for _ in range(N)]
    for t in range(N):
        for a in anc[t]: desc[a].add(t)
    return N,prereq,anc,desc,layers,width

def sample_student(N,prereq,layers,width,rng):
    lvl=rng.uniform(1,layers)  # ability
    known=np.zeros(N,bool)
    for t in range(N):
        L=t//width
        p=1/(1+np.exp((L-lvl)*1.5))
        known[t]=all(known[q] for q in prereq[t]) and rng.random()<p
    return known

def answer(known,t,slip,guess,rng):
    return rng.random()<(1-slip if known[t] else guess)

def run(strategy,G,known,prior,slip,guess,rng,confirm=False):
    N,prereq,anc,desc,_,_=G
    state=np.full(N,-1); q=0
    while (state<0).any():
        un=np.where(state<0)[0]
        if strategy=='random_noinf':
            t=rng.choice(un)
        elif strategy=='random_prop':
            t=rng.choice(un)
        else: # greedy bisection
            best=None;bv=-1
            for t0 in un:
                a=sum(1 for x in anc[t0] if state[x]<0)+1
                d=sum(1 for x in desc[t0] if state[x]<0)+1
                v=prior[t0]*a+(1-prior[t0])*d
                v=min(a,d)
                if v>bv: bv=v;best=t0
            t=best
        if confirm:
            r=[answer(known,t,slip,guess,rng),answer(known,t,slip,guess,rng)];q+=2
            if r[0]!=r[1]: r.append(answer(known,t,slip,guess,rng));q+=1
            ok=sum(r)>=2
        else:
            ok=answer(known,t,slip,guess,rng);q+=1
        if strategy=='random_noinf':
            state[t]=ok
        elif ok:
            state[t]=1
            for x in anc[t]:
                if state[x]<0: state[x]=1
        else:
            state[t]=0
            for x in desc[t]:
                if state[x]<0: state[x]=0
    return q,(state==known).mean()



In [2]:
rng=np.random.default_rng(0)
G=make_graph(); N=G[0]
students=[sample_student(N,G[1],G[4],G[5],rng) for _ in range(300)]
prior=np.mean(students,axis=0)
print(f"{N} topics; average student knows {prior.mean():.0%} of them")

150 topics; average student knows 54% of them


## 1. Questions needed with no answer noise

In [3]:
for strat in ['random_noinf','random_prop','greedy']:
    qs=[run(strat,G,s,prior,0,0,rng)[0] for s in students[:150]]
    print(f"{strat:13s}: {np.mean(qs):6.1f} questions")

random_noinf :  150.0 questions
random_prop  :   43.3 questions


greedy       :   38.1 questions


- Testing every topic: **150 questions**.
- Random order + graph inference: **~44**.
- Greedy choice (ask the topic whose answer resolves the most topics either way, a bisection rule): **~38**, about **4x fewer** than testing everything.

## 2. Add realistic noise: 10% slip, 10% guess

In [4]:
def evalc(fn,n=150):
    qs=[];acc=[]
    for s in students[:n]:
        q,a=fn(s); qs.append(q); acc.append(a)
    return np.mean(qs),np.mean(acc)
for name,fn in [('test every topic',lambda s:run('random_noinf',G,s,prior,.1,.1,rng)),
                ('greedy + inference',lambda s:run('greedy',G,s,prior,.1,.1,rng)),
                ('greedy, confirm every answer',lambda s:run('greedy',G,s,prior,.1,.1,rng,confirm=True))]:
    q,a=evalc(fn); print(f"{name:30s} questions {q:5.1f}  topics classified correctly {a:.1%}")

test every topic               questions 150.0  topics classified correctly 89.9%


greedy + inference             questions  37.0  topics classified correctly 94.3%


greedy, confirm every answer   questions  82.9  topics classified correctly 98.6%


With hard inference, one slip on a high-leverage topic can mislabel a whole subtree. Re-asking *every* question fixes accuracy but roughly doubles the test length.

## 3. Cheap fix: confirm only high-stakes answers

Re-ask a question only when its answer would resolve more than *k* other topics.

In [5]:
def run_stakes(known,k,slip=.1,guess=.1):
    N,prereq,anc,desc,_,_=G
    state=np.full(N,-1); q=0
    while (state<0).any():
        un=np.where(state<0)[0]; best=None;bv=-1;ba=bd=0
        for t0 in un:
            a=sum(1 for x in anc[t0] if state[x]<0)+1
            d=sum(1 for x in desc[t0] if state[x]<0)+1
            if min(a,d)>bv: bv=min(a,d);best=t0;ba,bd=a,d
        t=best; r=[answer(known,t,slip,guess,rng)]; q+=1
        if (ba if r[0] else bd)>k:
            r.append(answer(known,t,slip,guess,rng)); q+=1
            if r[0]!=r[1]: r.append(answer(known,t,slip,guess,rng)); q+=1
        ok=sum(r)*2>len(r)
        state[t]=int(ok)
        for x in (anc[t] if ok else desc[t]):
            if state[x]<0: state[x]=int(ok)
    return q,(state==known).mean()
for k in [3,5,10]:
    q,a=evalc(lambda s:run_stakes(s,k)); print(f"confirm if stake > {k:2d}: questions {q:5.1f}  correct {a:.1%}")

confirm if stake >  3: questions  48.5  correct 97.0%


confirm if stake >  5: questions  44.4  correct 96.0%


confirm if stake > 10: questions  40.6  correct 95.9%


**Result (this simulation):** confirming only high-stakes answers (k = 3) cuts misclassified topics roughly in half (about 5.7% to 3.0%) for about 11 extra questions, versus about 46 extra questions to confirm every answer.

## Caveats and the better version
- Synthetic graph and a simple slip/guess model; real diagnostics surely have their own safeguards.
- The stronger approach is soft inference: keep a probability per topic and let evidence propagate with weights instead of hard labels, so a single slip shifts beliefs instead of flipping a subtree. I'd be glad to test that against the hard rule.

By Jahangir Mammadli · mammadli.dev